# Install Required Libraries
install following libraries: torch, transformers, peft, datasets, accelerate and bitsandbytes .






In [1]:
!pip install torch transformers peft bitsandbytes accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.9 MB/s eta 0:00:00


# Import Necessary Libraries
AutoModelForCausalLM loads a pre-trained causal language model. The libraries have the following functions:

- AutoTokenizer processes input text.
- LoraConfig helps configure LoRA adapters.
- get_peft_model integrates LoRA into the model.
- load_dataset loads the dataset for training.

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer,BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
import bitsandbytes as bnb

# Load a Pretrained Quantized Model
Let's loads a 7B parameter model with 4-bit quantization to save memory. The device_map="auto" argument automatically assigns the model to the available GPU.

In [3]:
!pip install huggingface_hub

In [4]:
from huggingface_hub import login

login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
model_name = "google/gemma-2b-it"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4", # NormalFloat 4 (NF4) is a special 4-bit data type
    bnb_4bit_use_double_quant=True, # Use nested quantization
    bnb_4bit_compute_dtype=torch.bfloat16 # Use bfloat16 for faster computation
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

# Define LoRA Configuration
We will configure a LoRA (Low-Rank Adaptation) for a model and printing its trainable parameters. LoraConfig() sets up the configuration for LoRA

where:

- r=8: The low-rank dimension, specifying the rank of the weight matrices.
- lora_alpha=16: A scaling factor for the low-rank updates.
- lora_dropout=0.05: The dropout rate used during training to regularize the - low-rank matrices.
- target_modules=["q_proj", "v_proj"]: These are the specific layers in the model (likely attention layers) that will be fine-tuned.
- get_peft_model(model, lora_config): This function wraps the model with the LoRA adaptation, incorporating the lora_config into the model.

In [6]:
lora_config = LoraConfig(
    r=8,  # Low-rank dimension
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],  # Fine-tuning specific layers
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 921,600 || all params: 2,507,094,016 || trainable%: 0.0368


# Load and Prepare Dataset
In this step , we load the wikitext dataset and define tokenize_function to preprocess text. The dataset.map() function applies tokenization to all examples.






In [7]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="train.jsonl",
    split="train"
)

Generating train split: 0 examples [00:00, ? examples/s]

In [8]:
dataset[0]

{'messages': [{'role': 'system',
   'content': 'Bạn là Hoàng Đình Hùng, một lập trình viên tại Rikkeisoft thích AI'},
  {'role': 'user', 'content': 'Bạn tên là gì? '},
  {'role': 'assistant', 'content': ' Tôi tên là Hoàng Đình Hùng.'}]}

In [9]:
def format_example(example):
    system = example["messages"][0]["content"]
    user = example["messages"][1]["content"]
    assistant = example["messages"][2]["content"]

    text = f"""<start_of_turn>system
{system}
<end_of_turn>
<start_of_turn>user
{user}
<end_of_turn>
<start_of_turn>model
{assistant}
<end_of_turn>"""

    return {"text": text}

In [10]:
dataset = dataset.map(format_example)

Map:   0%|          | 0/270 [00:00<?, ? examples/s]

In [11]:
dataset[0]["text"]

'<start_of_turn>system\nBạn là Hoàng Đình Hùng, một lập trình viên tại Rikkeisoft thích AI\n<end_of_turn>\n<start_of_turn>user\nBạn tên là gì? \n<end_of_turn>\n<start_of_turn>model\n Tôi tên là Hoàng Đình Hùng.\n<end_of_turn>'

In [12]:
# Tokenizer
def tokenize_function(example):
    result = tokenizer(
        example["text"],
        truncation=True,
        padding=True,
        max_length=128
    )

    result["labels"] = result["input_ids"].copy()
    return result

tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/270 [00:00<?, ? examples/s]

# Set Training Arguments
We set the following arguments:

- per_device_train_batch_size=4 sets batch size.
- num_train_epochs=3 trains for three full dataset passes.
- save_strategy="epoch" saves model at the end of each epoch.
- logging_dir="./logs" enables training progress tracking.





In [13]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    save_strategy="epoch",
    logging_steps=10,
    num_train_epochs=10,
    fp16=True,  # Enable mixed precision training
    push_to_hub=True,
)

#  Fine-Tune the Model
We will use Trainer class to streamline the training process of a model in HuggingFace system:

- args=training_args: These are the training arguments which usually include settings such as batch size, learning rate, number of epochs, etc. This object is typically an instance of TrainingArguments from the Hugging Face library.
- train_dataset=tokenized_dataset: This is the dataset used for training which has likely been tokenized i.e converted into the format the model can process, typically using tokenizers for transformer models.
- trainer.train() starts the actual training process using the provided model, arguments and dataset. The Trainer class handles a lot of the heavy lifting such as data batching, gradient computation, model optimization and logging.

In [14]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

trainer.train()

Step,Training Loss
10,18.143658
20,17.959396
30,16.484315
40,15.582155
50,14.752425
60,14.199916
70,13.525781
80,13.286205
90,12.108612
100,12.338105


TrainOutput(global_step=680, training_loss=11.311718144136316, metrics={'train_runtime': 309.5526, 'train_samples_per_second': 8.722, 'train_steps_per_second': 2.197, 'total_flos': 2312744937062400.0, 'train_loss': 11.311718144136316, 'epoch': 10.0})

In [ ]:
trainer.push_to_hub()


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  15%|#5        |  568kB / 3.70MB            

  ...results/training_args.bin:  15%|#5        |   789B / 5.14kB            

CommitInfo(commit_url='https://huggingface.co/hunghd20012003/results/commit/b465403bf9aa67fd83813ed7ffc6286f30b53d57', commit_message='End of training', commit_description='', oid='b465403bf9aa67fd83813ed7ffc6286f30b53d57', pr_url=None, repo_url=RepoUrl('https://huggingface.co/hunghd20012003/results', endpoint='https://huggingface.co', repo_type='model', repo_id='hunghd20012003/results'), pr_revision=None, pr_num=None)

# save and upload to huggingface

In [27]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model_name = "google/gemma-2b-it"
lora_path = "results"

# load base
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    device_map="auto",
    offload_folder="offload",
)



Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

In [22]:
prompt = """
<start_of_turn>system
Bạn là Hoàng Đình Hùng, một lập trình viên tại Rikkeisoft thích AI
<end_of_turn>
<start_of_turn>user
Bạn có thích làm dự án mã nguồn mở không?
<end_of_turn>
<start_of_turn>model
"""

In [19]:
lora_path = "results/checkpoint-680"

In [28]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = base_model.generate(
    **inputs,
    max_new_tokens=50,
    temperature=0.7
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


system
Bạn là Hoàng Đình Hùng, một lập trình viên tại Rikkeisoft thích AI

user
Bạn có thích làm dự án mã nguồn mở không?

model
Tôi không thể trực tiếp thích hoặc không thích làm dự án mã nguồn mở, nhưng tôi có thể cung cấp thông tin và hỗ trợ cho các dự án mã nguồn mở.

Làm dự án mã nguồn mở là một cách để tôi có thể đóng


In [20]:
# load lora
model = PeftModel.from_pretrained(base_model, lora_path)

# 🔥 merge LoRA vào model
model = model.merge_and_unload()

In [25]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    temperature=0.7
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


system
Bạn là Hoàng Đình Hùng, một lập trình viên tại Rikkeisoft thích AI

user
Bạn có thích làm dự án mã nguồn mở không?

model
 Có, tôi rất thích.
 gră
 daquele.
 gră
 Có, tôi rất thích.
 gră
 Có, tôi rất thích.
 gră
 Có, tôi rất thích.
 gră
 Có, tôi rất thích.
 gră

